In [ ]:
# DO NOT MODIFY THIS CELL

from abc import ABC, abstractmethod  

class AbstractSearchInterface(ABC):
    '''
    Abstract class to support search/insert operations (plus underlying data structure)
    
    '''
        
    @abstractmethod
    def insertElement(self, element):     
        '''
        Insert an element in a search tree
            Parameters:
                    element: string to be inserted in the search tree (string)

            Returns:
                    "True" after successful insertion, "False" if element is already present (bool)
        '''
        
        pass 
    

    @abstractmethod
    def searchElement(self, element):
        '''
        Search for an element in a search tree
            Parameters:
                    element: string to be searched in the search tree (string)

            Returns:
                    "True" if element is found, "False" otherwise (bool)
        '''

        pass

In [ ]:
"""
LLRB Tree Implementation: 

A LLRB tree (Left-Leaning Red-Black tree) is a balanced search tree that maintains the following extra invariants:
- Each node has an assigned colour (red or black), and has at most two children.
- The root of the tree is coloured black.
- A red node cannot have a red child.
- The black-height of all leaf nodes is constant (black-height is the number of black nodes between the leaf and the root).
- The right child of any node is coloured black (this is the left-leaning invariant).

The invariants are maintained through combinations of flip and rotate methods.
LLRB trees are isomorphic to 2-3 trees. Nodes with a red left child can be thought of as 3-nodes,
and nodes without a red link are 2-nodes. It is useful to think of this connection when reading the
deletion logic.

Searches and insertions are O(log n) operations. The tree's depth is bounded between
log_2(n + 1) and 2 * log_2(n + 1) inclusive, where n is the number of nodes in the tree.
"""


class LLRBNode:
    """A node in a LLRB tree. Has an extra property: colour"""

    def __init__(self, key):
        self.key: str = key
        self.left: LLRBNode | None = None
        self.right: LLRBNode | None = None
        self.colour: bool = True  # True for red; False for black


class LLRBTree(AbstractSearchInterface):
    """A Left-Leaning Red-Black tree with search, insert and delete operations."""

    def __init__(self, root=LLRBNode | None):
        self.root = root

    def __is_red(self, node: LLRBNode | None) -> bool:
        """Returns True if the specified node exists and is red."""
        return node is not None and node.colour

    def __rotate_left(self, node: LLRBNode) -> LLRBNode:
        """Performs a left rotation about a given node."""
        right_child = node.right
        node.right = right_child.left
        right_child.left = node
        right_child.colour = node.colour
        node.colour = True
        return right_child

    def __rotate_right(self, node: LLRBNode) -> LLRBNode:
        """Performs a right rotation about a given node."""
        left_child = node.left
        node.left = left_child.right
        left_child.right = node
        left_child.colour = node.colour
        node.colour = True
        return left_child

    def __flip_colours(self, node: LLRBNode) -> LLRBNode:
        """Flips the colours of a given node and its two children."""
        node.colour = not node.colour
        node.left.colour = not node.left.colour
        node.right.colour = not node.right.colour
        return node

    def __fix_upward(self, node: LLRBNode) -> LLRBNode:
        """Fixes a node that breaks an invariant using rotation and flip operations.
        The rotations fix the error locally; the flip pushes the error up the tree.
        """
        # Case 1: Right child is red and left child is black — rotate left.
        if self.__is_red(node.right) and not self.__is_red(node.left):
            node = self.__rotate_left(node)
        # Case 2: Left child and its left child are both red — rotate right.
        if self.__is_red(node.left) and self.__is_red(node.left.left):
            node = self.__rotate_right(node)
        # Case 3: Both children are red — flip colours.
        if self.__is_red(node.left) and self.__is_red(node.right):
            node = self.__flip_colours(node)
        return node

    # --- Search ---

    def __search(self, element: str, node: LLRBNode | None) -> bool:
        """Recursively searches the subtree rooted at node for element.
        Returns True if found.
        """
        if node is None:
            return False
        elif node.key == element:
            return True
        elif element < node.key:
            return self.__search(element, node.left)
        else:
            return self.__search(element, node.right)

    def searchElement(self, element: str) -> bool:
        """Searches the tree for element. Returns True if found; False otherwise."""
        return self.__search(element, self.root)

    # --- Insert ---

    def __insert(self, element: str, node: LLRBNode | None) -> LLRBNode:
        """Recursively inserts an element into the LLRB subtree rooted at node.
        Duplicates are assumed to have been filtered by the caller.
        """
        if node is None:
            return LLRBNode(element)

        if element < node.key:
            node.left = self.__insert(element, node.left)
        elif element > node.key:
            node.right = self.__insert(element, node.right)

        return self.__fix_upward(node)

    def insertElement(self, element: str) -> bool:
        """Inserts an element into the tree.
        Returns False if the element was already present; True otherwise.
        """
        if self.__search(element, self.root):
            return False
        self.root = self.__insert(element, self.root)
        self.root.colour = False
        return True

    # --- Delete ---

    def __move_red_left(self, node: LLRBNode) -> LLRBNode:
        """Turns node's left child (a 2-node) into a 3-node by redistributing
        from the right child (if it has a red left link), or by merging into
        a temporary 4-node.
        """
        self.__flip_colours(node)
        if self.__is_red(node.right.left):
            node.right = self.__rotate_right(node.right)
            node = self.__rotate_left(node)
            self.__flip_colours(node)
        return node

    def __move_red_right(self, node: LLRBNode) -> LLRBNode:
        """Turns node's right child (a 2-node) into a 3-node by merging into
        a temporary 4-node, or by redistributing from the left child.
        Simpler than move_red_left because the left sibling's red link already leans left.
        """
        self.__flip_colours(node)
        if self.__is_red(node.left.left):
            node = self.__rotate_right(node)
            self.__flip_colours(node)
        return node

    def __get_minimum_key(self, node: LLRBNode) -> str:
        """Returns the smallest key in the subtree rooted at node."""
        while node.left is not None:
            node = node.left
        return node.key

    def __delete(self, element: str, node: LLRBNode) -> LLRBNode:
        """Recursively deletes element from the LLRB subtree rooted at node."""
        if element < node.key:
            # If descending into a 2-node, move a red link to the left child.
            if not self.__is_red(node.left) and not self.__is_red(node.left.left):
                node = self.__move_red_left(node)
            node.left = self.__delete(element, node.left)
        else:
            # Rotate right to push redness to the right side, making the current node
            # safe to delete and simplifying the rightward descent.
            if self.__is_red(node.left):
                node = self.__rotate_right(node)
            # Delete the node if found at the bottom of the tree.
            if node.right is None and node.key == element:
                return None
            # If descending into a 2-node, move a red link to the right child.
            if not self.__is_red(node.right) and not self.__is_red(node.right.left):
                node = self.__move_red_right(node)
            # Node matches but is not a leaf — replace with in-order successor and
            # delete the successor from the right subtree.
            if node.key == element:
                node.key = self.__get_minimum_key(node.right)
                node.right = self.__delete(node.key, node.right)
            else:
                node.right = self.__delete(element, node.right)

        return self.__fix_upward(node)

    def deleteElement(self, element: str) -> bool:
        """Deletes element from the tree.
        Returns False if the element was not present; True otherwise.
        """
        if not self.__search(element, self.root):
            return False
        self.root = self.__delete(element, self.root)
        if self.root is not None:
            self.root.colour = False
        return True